In [2]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import torch
import sentence_transformers

print("torch:", torch.__version__, "| CUDA GPU:", torch.cuda.is_available())
print("sentence-transformers:", sentence_transformers.__version__)

DATA = Path("data")
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(f"CREATE VIEW articles AS SELECT * FROM '{(DATA / 'articles.parquet').as_posix()}'")
print(con.execute("SELECT COUNT(*) FROM articles").fetchone()[0], "articles")

torch: 2.11.0+cpu | CUDA GPU: False
sentence-transformers: 2.5.1
105542 articles


In [3]:
arts = con.execute("""
    SELECT article_id, prod_name, product_type_name, colour_group_name,
           graphical_appearance_name, index_group_name, department_name,
           COALESCE(detail_desc, '') AS detail_desc
    FROM articles
    ORDER BY article_id
""").df()

arts["text"] = (
    arts["prod_name"] + ". "
    + arts["product_type_name"] + ", " + arts["colour_group_name"] + ", "
    + arts["graphical_appearance_name"] + ". "
    + arts["index_group_name"] + ", " + arts["department_name"] + ". "
    + arts["detail_desc"]
).str.strip()

print("Missing descriptions:", (arts["detail_desc"] == "").sum())
print("Words per text: median", arts["text"].str.split().str.len().median(),
      "| max", arts["text"].str.split().str.len().max())
print()
for t in arts["text"].head(3):
    print("-", t, "\n")

Missing descriptions: 416
Words per text: median 32.0 | max 130

- Strap top. Vest top, Black, Solid. Ladieswear, Jersey Basic. Jersey top with narrow shoulder straps. 

- Strap top. Vest top, White, Solid. Ladieswear, Jersey Basic. Jersey top with narrow shoulder straps. 

- Strap top (1). Vest top, Off White, Stripe. Ladieswear, Jersey Basic. Jersey top with narrow shoulder straps. 



In [4]:
import time
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
print("Embedding size:", model.get_sentence_embedding_dimension())
print("Max tokens read:", model.max_seq_length)

start = time.time()
test = model.encode(arts["text"].head(1000).tolist(), batch_size=64,
                    normalize_embeddings=True, show_progress_bar=False)
secs = time.time() - start
print(f"1,000 texts in {secs:.1f}s")
print(f"Estimated time for all {len(arts):,}: {secs * len(arts) / 1000 / 60:.1f} minutes")

README.md: 0.00B [00:00, ?B/s]

Embedding size: 384
Max tokens read: 256
1,000 texts in 4.6s
Estimated time for all 105,542: 8.1 minutes


In [5]:
start = time.time()
emb = model.encode(arts["text"].tolist(), batch_size=64,
                   normalize_embeddings=True, show_progress_bar=False)
print(f"Done in {(time.time() - start) / 60:.1f} minutes")
print("Shape:", emb.shape, "| dtype:", emb.dtype)

np.save(DATA / "article_emb.npy", emb.astype(np.float32))
np.save(DATA / "article_emb_ids.npy", arts["article_id"].to_numpy(dtype="U10"))
print("Saved.")

# Sanity check: every vector should have length 1
norms = np.linalg.norm(emb, axis=1)
print(f"Vector lengths: min {norms.min():.4f}, max {norms.max():.4f}")

Done in 13.7 minutes
Shape: (105542, 384) | dtype: float32
Saved.
Vector lengths: min 1.0000, max 1.0000


In [6]:
arts["product_code"] = arts["article_id"].str[:7]

def similar(article_id, n=5):
    i = arts.index[arts["article_id"] == article_id][0]
    scores = emb @ emb[i]
    order = np.argsort(-scores)
    out = []
    for j in order:
        if arts.at[j, "product_code"] == arts.at[i, "product_code"]:
            continue
        out.append((round(float(scores[j]), 3), arts.at[j, "prod_name"],
                    arts.at[j, "product_type_name"], arts.at[j, "colour_group_name"]))
        if len(out) == n:
            break
    return out

for aid in ["0108775015", "0706016001", "0610776002"]:
    row = arts[arts["article_id"] == aid].iloc[0]
    print(f"\n>>> {row['prod_name']} | {row['product_type_name']} | {row['colour_group_name']}")
    for s in similar(aid):
        print("   ", s)


>>> Strap top | Vest top | Black
    (0.952, 'Strap Top.', 'Vest top', 'Black')
    (0.945, 'Strap top 3-pack', 'Vest top', 'Black')
    (0.942, 'Strap Top 2 pack', 'Vest top', 'Black')
    (0.937, 'Strap Top.', 'Vest top', 'White')
    (0.928, 'Strap Top 2 pack', 'Vest top', 'Grey')

>>> Jade HW Skinny Denim TRS | Trousers | Black
    (0.962, 'Jade HW Skinny Belt Denim TRS', 'Trousers', 'Black')
    (0.958, 'Jade HW Skinny Belt Denim TRS', 'Trousers', 'Beige')
    (0.955, 'Jade HW Skinny Belt Denim TRS', 'Trousers', 'Greenish Khaki')
    (0.935, 'Jade Trash HW Skinny Denim TRS', 'Trousers', 'Black')
    (0.934, 'Jade Trash HW Skinny Denim TRS', 'Trousers', 'Blue')

>>> Tilly (1) | T-shirt | Black
    (0.96, 'Tilly.', 'T-shirt', 'Black')
    (0.95, 'Tilly.', 'T-shirt', 'Dark Red')
    (0.949, 'Tilly.', 'T-shirt', 'White')
    (0.944, 'Tilly.', 'T-shirt', 'Light Yellow')
    (0.943, 'Tilly.', 'T-shirt', 'Light Beige')


In [7]:
from metrics import apk, mapk

con.execute(f"CREATE VIEW tx AS SELECT * FROM '{(DATA / 'sample_transactions.parquet').as_posix()}'")

labels = con.execute(f"SELECT customer_id, bought FROM '{(DATA / 'labels_valid.parquet').as_posix()}'").df()
actual = {c: list(b) for c, b in zip(labels["customer_id"], labels["bought"])}

pool_ids = con.execute("SELECT DISTINCT article_id FROM tx WHERE w BETWEEN 2 AND 3").df()["article_id"]
id_to_row = {a: i for i, a in enumerate(arts["article_id"])}
pool_rows = np.array([id_to_row[a] for a in pool_ids])
pool_emb = emb[pool_rows]
pool_ids = pool_ids.to_numpy()

print(f"Validation customers: {len(actual):,}")
print(f"Candidate pool: {len(pool_ids):,} items  | pool matrix {pool_emb.shape}")

Validation customers: 14,471
Candidate pool: 14,686 items  | pool matrix (14686, 384)


In [8]:
top2 = con.execute("""
    SELECT article_id FROM tx WHERE w BETWEEN 2 AND 3
    GROUP BY article_id ORDER BY COUNT(DISTINCT customer_id) DESC, article_id LIMIT 12
""").df()["article_id"].tolist()

pdf = con.execute("""
    SELECT customer_id, LIST(article_id ORDER BY last_buy DESC, n DESC, article_id) AS items
    FROM (SELECT customer_id, article_id, MAX(t_dat) AS last_buy, COUNT(*) AS n
          FROM tx WHERE w BETWEEN 2 AND 9 GROUP BY customer_id, article_id)
    GROUP BY customer_id
""").df()
past = {c: list(i) for c, i in zip(pdf["customer_id"], pdf["items"])}

def fill(items, filler, k=12):
    out = list(items[:k])
    for f in filler:
        if len(out) >= k: break
        if f not in out: out.append(f)
    return out

# Similar-item lists for customers with history
custs = [c for c in actual if c in past]
sim_lists = {}
for s in range(0, len(custs), 2000):
    cs = custs[s:s + 2000]
    prof = np.stack([emb[[id_to_row[a] for a in past[c]]].mean(axis=0) for c in cs])
    prof /= np.linalg.norm(prof, axis=1, keepdims=True)
    S = prof @ pool_emb.T
    top = np.argpartition(-S, 50, axis=1)[:, :50]
    for k, c in enumerate(cs):
        idx = top[k][np.argsort(-S[k, top[k]])]
        sim_lists[c] = [pool_ids[j] for j in idx]

def build(c, n_sim):
    items = list(past.get(c, [])[:12])
    added = 0
    for a in sim_lists.get(c, []):
        if added >= n_sim or len(items) >= 12: break
        if a not in items:
            items.append(a); added += 1
    return fill(items, top2)

ap = {}
for n_sim in [0, 2, 4, 12]:
    ap[n_sim] = np.array([apk(actual[c], build(c, n_sim)) for c in actual])
    print(f"n_sim={n_sim:<3} MAP@12 = {ap[n_sim].mean():.5f}")

n_sim=0   MAP@12 = 0.02463
n_sim=2   MAP@12 = 0.02515
n_sim=4   MAP@12 = 0.02517
n_sim=12  MAP@12 = 0.02514


In [9]:
def paired_gain(ap_new, ap_old, n_boot=1000, seed=42):
    diff = ap_new - ap_old
    rng = np.random.default_rng(seed)
    boot = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(n_boot)]
    low, high = np.percentile(boot, [2.5, 97.5])
    return diff.mean(), low, high, (diff > 0).sum(), (diff < 0).sum()

gain, low, high, better, worse = paired_gain(ap[4], ap[0])
print(f"Similar items (4) vs bought-before only")
print(f"Mean gain: {gain:+.5f}   95% range: {low:+.5f} to {high:+.5f}")
print(f"Customers better: {better:,}   worse: {worse:,}")
print("Real gain" if low > 0 else "Not clearly real (range includes 0)")

Similar items (4) vs bought-before only
Mean gain: +0.00053   95% range: +0.00021 to +0.00088
Customers better: 133   worse: 245
Real gain


## Text embeddings and "similar items"

I turned each product into one line of text (name, type, colour, pattern, department and description) and encoded all 105,542 products with the all-MiniLM-L6-v2 sentence-transformer. Each product became a list of 384 numbers. On my CPU this took 13.7 minutes, much longer than my 8-minute estimate from a 1,000-item test, probably because later products have longer descriptions. I saved the vectors so I never have to run it again.

The neighbours made sense when I looked at them: strap tops next to strap tops, skinny jeans next to skinny jeans. But many neighbours were really the same product under a different code, like "Strap Top." and "Strap top". So the model is good at finding near-copies and less good at finding something new.

To use it for recommendations, I averaged the vectors of each customer's last 8 weeks of purchases and looked for the closest items among things that sold in the previous two weeks. I added those after the customer's past purchases and before the best-sellers. Adding 4 similar items moved MAP@12 from 0.02463 to 0.02517. A paired bootstrap put the gain between +0.00021 and +0.00088, so it's real.

It's smaller than the gain from "bought together" (+0.00091), and more customers got worse (245) than better (133). I think the similar items push out best-sellers that many people would have picked up, while a smaller group gets a bigger win. So embeddings help, but a simple co-purchase count still does better here.